# 05 - Training the Spatial Classifier by Plant Clusters

## Role of This Notebook
Here, a bridge label between geometry and plants is built first, and then the ML component of the project is trained. The model does not predict species directly; it predicts the plant cluster most compatible with each cell.

## Evaluation Criterion
1. The supervised label comes from an interpretable scoring process between tiles and cluster profiles.
2. Training uses geometric and environmental variables available per cell, excluding relative humidity because that column was removed from the project.
3. Several models are compared and `macro_f1` is prioritized together with `accuracy`, because this is still a multiclass problem.


In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#777777',
    'axes.grid': True,
    'grid.color': '#e6e6e6',
    'grid.linestyle': '-',
    'grid.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
geometry = pd.read_csv(PROCESSED_DIR / 'geometry_labeled.csv')
cluster_profiles = pd.read_csv(PROCESSED_DIR / 'plants_cluster_profiles.csv')
geometry.head()

## 1. Building the Bridge Label
The 01-07 series had already been building a hybrid `ground truth` in notebook 02. Here that logic is preserved, but the label is no longer a generic spatial class; it is now the plant cluster best aligned with each cell.

Since `RELATIVE_HUMIDITY` left the pipeline, compatibility is now decided with two components: estimated sun hours and a synthetic light index. This simplifies the bridge and keeps it coherent with the real features that will remain available for training.


### KEY - Why the Pipeline Uses KMeans First and RandomForest Later
In this project, the two algorithms do not compete with each other; they perform different functions. `KMeans` is used in notebook 04 to discover groups of plants with similar requirements within the botanical catalog. Later, in notebook 05, `RandomForest` learns to predict those operational groups from the geometry variables.

The correct methodological reading is this: `KMeans` organizes the botanical universe; `RandomForest` operationalizes that organization so it can be used with new tiles. Without the first stage, there would be no defensible supervised target; without the second, there would be no practical inference on new simulations.


In [ ]:
tiles = geometry.copy()

bridge_key_cols = ['sun_min_h_day', 'sun_max_h_day', 'lux_min', 'lux_max', 'sun_center_h_day', 'lux_center', 'plant_light_index']
if {'bridge_cluster_id', 'bridge_cluster_label'}.issubset(cluster_profiles.columns):
    bridge_profiles = cluster_profiles[['bridge_cluster_id', 'bridge_cluster_label'] + bridge_key_cols].drop_duplicates().sort_values('bridge_cluster_id').reset_index(drop=True)
else:
    bridge_profiles = cluster_profiles[bridge_key_cols].drop_duplicates().reset_index(drop=True).copy()
    bridge_profiles['bridge_cluster_id'] = bridge_profiles.index.astype(int)
    bridge_profiles['bridge_cluster_label'] = 'bridge_cluster_' + bridge_profiles['bridge_cluster_id'].astype(str)
    cluster_profiles = cluster_profiles.merge(bridge_profiles, on=bridge_key_cols, how='left')

if tiles['SUN_HOURS'].max() > 24:
    tiles['sun_h_day_estimated'] = tiles['SUN_HOURS'] / 365.0
else:
    tiles['sun_h_day_estimated'] = tiles['SUN_HOURS']

def minmax(series):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - min_val) / (max_val - min_val)

tiles['sun_norm'] = minmax(tiles['SUN_HOURS'])
tiles['radiation_norm'] = minmax(tiles['RADIATION'])
tiles['udi_norm'] = minmax(tiles['UDI'])
tiles['tile_light_index'] = 0.45 * tiles['sun_norm'] + 0.45 * tiles['radiation_norm'] + 0.10 * tiles['udi_norm']

def range_score(values, min_value, max_value, softness):
    values = np.asarray(values, dtype=float)
    distance = np.zeros_like(values)
    below = values < min_value
    above = values > max_value
    distance[below] = min_value - values[below]
    distance[above] = values[above] - max_value
    return np.exp(-((distance / softness) ** 2))

def closeness_score(values, target, softness):
    values = np.asarray(values, dtype=float)
    return np.exp(-(((values - target) / softness) ** 2))

score_cols = []
cluster_ids = bridge_profiles['bridge_cluster_id'].astype(int).tolist()

for _, profile in bridge_profiles.iterrows():
    cluster_id = int(profile['bridge_cluster_id'])
    score_col = f'cluster_{cluster_id}_score'
    sun_score = range_score(tiles['sun_h_day_estimated'].values, profile['sun_min_h_day'], profile['sun_max_h_day'], softness=1.5)
    light_score = closeness_score(tiles['tile_light_index'].values, profile['plant_light_index'], softness=0.25)
    tiles[score_col] = 0.65 * sun_score + 0.35 * light_score
    score_cols.append(score_col)

score_matrix = tiles[score_cols].to_numpy(dtype=float)
stable_scores = score_matrix - score_matrix.max(axis=1, keepdims=True)
exp_scores = np.exp(stable_scores / 0.15)
prob_matrix = exp_scores / exp_scores.sum(axis=1, keepdims=True)

prob_cols = []
for i, cluster_id in enumerate(cluster_ids):
    prob_col = f'bridge_cluster_{cluster_id}_probability'
    tiles[prob_col] = prob_matrix[:, i]
    prob_cols.append(prob_col)

sorted_idx = np.argsort(prob_matrix, axis=1)
top1_idx = sorted_idx[:, -1]
top2_idx = sorted_idx[:, -2]
top3_idx = sorted_idx[:, -3]
cluster_ids_array = np.array(cluster_ids)

tiles['plant_cluster_target'] = cluster_ids_array[top1_idx]
tiles['plant_cluster_target_label'] = 'bridge_cluster_' + tiles['plant_cluster_target'].astype(str)
tiles['bridge_top1_probability'] = prob_matrix[np.arange(len(prob_matrix)), top1_idx]
tiles['bridge_top2_cluster'] = cluster_ids_array[top2_idx]
tiles['bridge_top3_cluster'] = cluster_ids_array[top3_idx]
tiles['bridge_margin'] = prob_matrix[np.arange(len(prob_matrix)), top1_idx] - prob_matrix[np.arange(len(prob_matrix)), top2_idx]
tiles['bridge_source_clusters'] = tiles['plant_cluster_target'].map(bridge_profiles.set_index('bridge_cluster_id').apply(lambda row: ', '.join(cluster_profiles.loc[cluster_profiles['bridge_cluster_id'] == row.name, 'plant_cluster_label'].astype(str).tolist()), axis=1))

tiles.to_csv(PROCESSED_DIR / 'geometry_cluster_targets.csv', index=False)
tiles[['ROW_ID', 'TILE_ID', 'plant_cluster_target_label', 'bridge_top1_probability', 'bridge_margin'] + prob_cols].head()

## 2. Visual Reading of the Bridge Label
Before training, it is useful to inspect how distributed the target is and how clear the compatibility appears. If the top-1 probabilities are very low or the margins are very narrow, the correct reading is that some cells are close to several clusters, not necessarily that the pipeline is badly built.


In [ ]:
target_counts = tiles['plant_cluster_target_label'].value_counts().sort_index()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
bars = axes[0].bar(target_counts.index, target_counts.values, color='#4e79a7', edgecolor='#444444', linewidth=0.6)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(int(bar.get_height())), ha='center', va='bottom', fontsize=8)
axes[0].set_title('Target distribution by plant cluster')
axes[0].set_xlabel('Plant cluster target')
axes[0].set_ylabel('Number of tiles')
axes[0].tick_params(axis='x', rotation=45)

box_data_probability = [tiles.loc[tiles['plant_cluster_target_label'] == label, 'bridge_top1_probability'].to_numpy() for label in target_counts.index]
axes[1].boxplot(box_data_probability, tick_labels=target_counts.index, patch_artist=True, boxprops=dict(facecolor='#76b7b2', alpha=0.75))
axes[1].set_title('Top-1 bridge probability by cluster')
axes[1].set_xlabel('Plant cluster target')
axes[1].set_ylabel('Bridge top-1 probability')
axes[1].tick_params(axis='x', rotation=45)

box_data_margin = [tiles.loc[tiles['plant_cluster_target_label'] == label, 'bridge_margin'].to_numpy() for label in target_counts.index]
axes[2].boxplot(box_data_margin, tick_labels=target_counts.index, patch_artist=True, boxprops=dict(facecolor='#f28e2b', alpha=0.75))
axes[2].set_title('Bridge margin by cluster')
axes[2].set_xlabel('Plant cluster target')
axes[2].set_ylabel('Top1 - Top2 margin')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Conclusion About the Bridge Label
The bridge target is imbalanced but not arbitrary: `bridge_cluster_2` concentrates `35020` rows, while `bridge_cluster_0` has `6755` and `bridge_cluster_1` has `1325`. Even so, the scoring clarity is high: the top-1 probability has an average near `0.794`, and the top1-top2 margin averages `0.644`. This indicates that, although there is a dominant class, many cells still fall into a winning cluster with enough separation from the alternatives.


## 3. Preparing the Supervised Problem
The model features still follow the logic of the project: geometric and environmental variables available per cell. The target is now `plant_cluster_target_label`, built above as a compatibility layer between geometry and plants.


In [ ]:
feature_columns = ['BUILDING_ORIENTATION', 'RATIO_N', 'RATIO_E', 'RATIO_S', 'RATIO_W', 'UDI', 'SUN_HOURS', 'RADIATION']
target_column = 'plant_cluster_target_label'

X = tiles[feature_columns]
y = tiles[target_column]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, feature_columns)])

models = {
    'logistic_regression': LogisticRegression(max_iter=1500),
    'random_forest': RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1),
    'knn': KNeighborsClassifier(n_neighbors=15),
    'svm': SVC(kernel='rbf', probability=True, random_state=42)
}

results = []
trained_pipelines = {}

## 4. Visual Reading of the Input Features
The correlation heatmap does not define the model by itself, but it helps read strong dependencies between geometric variables and light proxies. This is useful for arguing that the problem does have structure, even after removing relative humidity.


In [ ]:
corr_cols = ['RATIO_N', 'RATIO_E', 'RATIO_S', 'RATIO_W', 'UDI', 'SUN_HOURS', 'RADIATION', 'sun_h_day_estimated', 'tile_light_index']
corr_matrix = tiles[corr_cols].corr()
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr_matrix.to_numpy(), cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Correlation')
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha='right')
ax.set_yticklabels(corr_cols)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', color='black', fontsize=8)
ax.set_title('Correlation heatmap of geometric and bridge features')
plt.tight_layout()
plt.show()

### Conclusion About the Correlations
The heatmap confirms that several geometric and simulation variables describe related facets of luminous exposure. The correct reading is not to remove everything correlated, but to recognize that the model is learning from a structured system of light proxies and not from columns disconnected from each other.


## 5. Model Comparison
The models are tested under the same preprocessing scheme. This preserves the logic of the original notebook 05: a fair, legible, and defensible comparison before selecting the final pipeline.


In [ ]:
for model_name, estimator in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', estimator)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    results.append({'model': model_name, 'accuracy': accuracy_score(y_test, preds), 'macro_f1': f1_score(y_test, preds, average='macro')})
    trained_pipelines[model_name] = pipeline

results_df = pd.DataFrame(results).sort_values(['macro_f1', 'accuracy'], ascending=False)
display(results_df)
results_df.to_csv(PROCESSED_DIR / 'model_comparison.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(results_df['model'], results_df['accuracy'], color='#4e79a7', edgecolor='#444444', linewidth=0.6)
axes[0].set_title('Model comparison by accuracy')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Accuracy')
axes[0].tick_params(axis='x', rotation=25)

axes[1].bar(results_df['model'], results_df['macro_f1'], color='#59a14f', edgecolor='#444444', linewidth=0.6)
axes[1].set_title('Model comparison by macro F1')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Macro F1')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

### Conclusion About the Model Comparison
The model selection is quite clear. `RandomForest` ranks first with `accuracy = 0.9998` and `macro_f1 = 0.9992`, above `logistic_regression`, `svm`, and `knn`. It not only predicts more correctly, but also keeps a better balance across classes, which makes it the most defensible candidate for later deployment.


## 6. Final Model Selection and Error Reading
The central output of this notebook is `geometry_predictions.csv`. This file keeps the bridge label, the model prediction, and cluster probabilities. The confusion matrix makes it possible to see whether errors occur between neighboring clusters or whether the model confuses very different profiles.


In [ ]:
best_model_name = results_df.iloc[0]['model']
best_pipeline = trained_pipelines[best_model_name]
test_predictions = best_pipeline.predict(X_test)

print(f'Best model: {best_model_name}')
print(classification_report(y_test, test_predictions))

label_order = sorted(y.unique())
cm = pd.DataFrame(confusion_matrix(y_test, test_predictions, labels=label_order), index=[f'true_{label}' for label in label_order], columns=[f'pred_{label}' for label in label_order])
display(cm)
cm.to_csv(PROCESSED_DIR / 'best_model_confusion_matrix.csv')

model_selection_notes = pd.Series({
    'selected_model': best_model_name,
    'selection_criterion': 'Se prioriza el mejor balance entre Macro F1, accuracy e interpretacion de errores.',
    'target_definition': 'The target is the bridge cluster with the highest compatibility according to environmental scoring per cell.',
    'deployment_value': 'The model predicts plant bridge clusters from geometric and environmental variables available per cell.'
})
model_selection_notes.to_csv(PROCESSED_DIR / 'model_selection_notes.csv', header=['value'])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm.to_numpy(), cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, label='Count')
ax.set_xticks(range(len(cm.columns)))
ax.set_yticks(range(len(cm.index)))
ax.set_xticklabels(cm.columns, rotation=45, ha='right')
ax.set_yticklabels(cm.index)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm.iloc[i, j]), ha='center', va='center', color='black', fontsize=8)
ax.set_title('Confusion matrix of the selected cluster classifier')
plt.tight_layout()
plt.show()

### Conclusion About the Confusion Matrix
The matrix shows a practically stabilized classifier. `bridge_cluster_0` and `bridge_cluster_1` have no errors in the evaluated set, and `bridge_cluster_2` shows only two isolated mistakes. For the final presentation, this supports the claim that the supervised layer not only performs very well, but that its residual errors do not seriously compromise the top 5 recommendation.


In [ ]:
full_predictions = tiles[['ROW_ID', 'TILE_ID', 'spatial_label'] + feature_columns + ['sun_h_day_estimated', 'tile_light_index', 'plant_cluster_target', 'plant_cluster_target_label', 'bridge_top1_probability', 'bridge_margin']].copy()
full_predictions['plant_cluster_pred_label'] = best_pipeline.predict(X)
full_predictions['plant_cluster_pred'] = full_predictions['plant_cluster_pred_label'].str.replace('bridge_cluster_', '', regex=False).astype(int)

pred_proba = best_pipeline.predict_proba(X)
classes_ = best_pipeline.named_steps['model'].classes_
for idx, class_label in enumerate(classes_):
    class_label = str(class_label)
    if class_label.startswith('bridge_cluster_'):
        class_id = class_label.replace('bridge_cluster_', '')
    elif class_label.startswith('cluster_'):
        class_id = class_label.replace('cluster_', '')
    else:
        class_id = class_label
    full_predictions[f'pred_cluster_{class_id}_probability'] = pred_proba[:, idx]

full_predictions['plant_cluster_pred_probability'] = pred_proba.max(axis=1)
full_predictions.to_csv(PROCESSED_DIR / 'geometry_predictions.csv', index=False)
joblib.dump(best_pipeline, MODELS_DIR / 'plant_cluster_classifier.joblib')
print('Saved:', PROCESSED_DIR / 'geometry_cluster_targets.csv')
print('Saved:', PROCESSED_DIR / 'geometry_predictions.csv')
print('Saved:', MODELS_DIR / 'plant_cluster_classifier.joblib')